In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import datasets as ds
from src.loss import HingeLoss, LogisticLoss, SquaredLoss
from src.classifiers import ClassifierLin

DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "report" / "Figures"
RESULTS_DIR = ROOT / "results"
sns.set_theme(style="whitegrid")

DATASETS = ["separable", "label_flip", "gaussian", "spambase"]
LOSSES = {"hinge": HingeLoss(), "logistic": LogisticLoss(), "squared": SquaredLoss()}
LAM, T, TOL = 0.05, 10000, 1e-10
RUNS = range(len(np.load(DATA_DIR / "spambase.npz")["train_idx"]))  

In [ ]:
def get_data(name, run):
    X_train, y_train, X_test, y_test, eta_test = ds.load_dataset(name, run, DATA_DIR)
    if name == "spambase":
        X_train, X_test = ds.standardize(X_train, X_test)
    return X_train, y_train, X_test, y_test, eta_test

In [3]:
K = 5
LAM_GRID = np.round(np.arange(0.01, 0.151, 0.01), 2)


def cv_error(loss, lam, X, y, folds, standardize):
    errors = []
    for k in range(K):
        val = folds[k]
        tr = np.concatenate([folds[j] for j in range(K) if j != k])
        X_tr, X_val = ds.standardize(X[tr], X[val]) if standardize else (X[tr], X[val])
        errors.append(ClassifierLin(loss, lam, T, TOL).fit(X_tr, y[tr]).error(X_val, y[val]))
    return np.mean(errors), np.std(errors, ddof=1) / np.sqrt(K)


rows = []
for run in RUNS:
    for name in DATASETS:
        X_train, y_train, *_ = ds.load_dataset(name, run, DATA_DIR)  
        folds = np.array_split(np.random.default_rng(run).permutation(len(y_train)), K)
        for loss_name, loss in LOSSES.items():
            for lam in LAM_GRID:
                err, se = cv_error(loss, lam, X_train, y_train, folds, name == "spambase")
                rows.append({"run": run, "dataset": name, "loss": loss_name, "lambda": lam, "cv error": err, "se": se})
cv = pd.DataFrame(rows)